In [ ]:
"""
PROJECT: Stock Market Data Collection (S&P 500)
End-to-End Time-Series ML System for Market Direction Prediction with Backtesting, MLOps, and LLM Explainability

WHAT THIS CODE DOES:
- Downloads historical S&P 500 stock market data
- Saves it to a CSV file so we don’t download it again and again
- Loads the data from the file if it already exists

WHY THIS IS IMPORTANT FOR MACHINE LEARNING:
- Machine Learning needs DATA first
- This project teaches you how to collect and store real-world data
"""

# STEP 0: Imports & Setup
import yfinance as yf      # Used to download stock market data
import pandas as pd        # Used to work with tables (like Excel)
import os                  # Used to check if files exist on your computer
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score



In [ ]:
"""
STEP 1: Data Ingestion

- Download S&P 500 data if not available locally
- Cache data to avoid repeated downloads
"""

if os.path.exists("sp500.csv"):
    sp500 = pd.read_csv("sp500.csv", index_col=0)
else:
    sp500 = yf.Ticker("^GSPC").history(period="max")
    sp500.to_csv("sp500.csv")

sp500.index = pd.to_datetime(sp500.index)


In [ ]:
"""
STEP 2: Data Cleaning & Visualization
"""

# Plot market trend
sp500.plot.line(y="Close", title="S&P 500 Closing Price")

# Remove unused columns
sp500.drop(columns=["Dividends", "Stock Splits"], inplace=True)

# Use modern market data
sp500 = sp500.loc["1990-01-01":].copy()



In [ ]:
# ============================
# STEP 3: Data Cleaning
# ============================

"""
WHY CLEAN DATA?

Imagine trying to learn with:
- Extra useless pages
- Missing information
- Confusing columns

Clean data = better learning 
"""

# Remove columns that are not useful for prediction
del sp500["Dividends"]
del sp500["Stock Splits"]


In [ ]:
"""
STEP 4: Create Prediction Target
FEATURE ENGINEERING = Teaching the model WHAT to look at

We create a TARGET:
- TRUE  → market goes UP tomorrow
- FALSE → market goes DOWN tomorrow
"""

# Convert index to datetime (important for time-based data)
sp500.index = pd.to_datetime(sp500.index)

# Create tomorrow's closing price
sp500["Tomorrow"] = sp500["Close"].shift(-1)

# Create target column (what we want to predict)
sp500["Target"] = sp500["Tomorrow"] > sp500["Close"]

# Use only modern market data
sp500 = sp500.loc["1990-01-01":].copy()

# Remove rows with missing values
sp500.dropna(inplace=True)


sp500["2020":"2024"] # Filter by Year
# Get Year / Month
sp500["Year"] = sp500.index.year
sp500["Month"] = sp500.index.month
# Plot Time Series Properly
sp500["Close"].plot(title="S&P 500 Over Time")

sp500["DayOfWeek"] = sp500.index.dayofweek
sp500["Year"] = sp500.index.year
sp500["Month"] = sp500.index.month





In [ ]:
sp500


In [ ]:
# ============================
# STEP 5: Baseline Model
# ============================

"""
STEP 5: Train Initial ML Model

WHAT IS MACHINE LEARNING DOING HERE?

The model looks at past prices and learns patterns
to guess whether the market will go UP or DOWN.
"""

from sklearn.ensemble import RandomForestClassifier

# Create the ML model
model = RandomForestClassifier(
    n_estimators=100,        # Number of decision trees (voters)
    min_samples_split=100,   # Prevents memorization
    random_state=1           # Same results every run
)
# Split data into training and testing
train = sp500.iloc[:-100]   # Old data (learning)
test = sp500.iloc[-100:]    # New data (exam)

# Features given to the model
predictors = ["Close", "Volume", "Open", "High", "Low"]

# Train the model
model.fit(train[predictors], train["Target"])

preds = model.predict(test[predictors])
precision_score(test["Target"], preds)


In [ ]:

"""
STEP 5: Basic Evaluation on Recent Data

GOAL:
- Test if our model is learning anything
- Measure precision (how often predictions are correct)
"""

# Use only modern data
sp500 = sp500.loc["1990-01-01":].copy()

from sklearn.metrics import precision_score

# Make predictions on test data
preds = model.predict(test[predictors])

# Convert predictions to a pandas Series
preds = pd.Series(preds, index=test.index)

# Measure precision
precision_score(test["Target"], preds)



In [ ]:
"""
STEP 6: Compare Predictions with Reality
Blue line → What actually happened
Orange line → What model guessed
"""

combined = pd.concat([test["Target"], preds], axis=1)
combined.plot()


In [ ]:
"""
STEP 7: Create a Prediction Function

WHY?
- Avoid repeating code
- Real ML projects use functions
"""

def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined


In [ ]:
"""
STEP 8: Backtesting

WHAT IS BACKTESTING?

We simulate the real world:
- Train on past data
- Predict the future
- Move forward in time
"""

def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []

    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    
    return pd.concat(all_predictions)



In [ ]:
# STEP 8: Run Backtest & Evaluate
predictions = backtest(sp500, model, predictors)

predictions["Predictions"].value_counts()

precision_score(
    predictions["Target"],
    predictions["Predictions"]
)


In [ ]:
"""
STEP 9: Feature Engineering with Rolling Windows

WHY?
Markets have memory.
Past days influence today.
Close_Ratio → Is price high or low compared to past?
Trend → How many UP days recently?
"""

horizons = [2, 5, 60, 250, 1000]
new_predictors = []

for horizon in horizons:
    rolling_averages = sp500.rolling(horizon).mean()
    
    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_averages["Close"]
    
    trend_column = f"Trend_{horizon}"
    sp500[trend_column] = sp500.shift(1).rolling(horizon).sum()["Target"]
    
    new_predictors += [ratio_column, trend_column]
    
# Remove rows with missing values
sp500 = sp500.dropna(subset=sp500.columns[sp500.columns != "Tomorrow"])


In [ ]:
# STEP 10: Probability-Aware Model - Probability Thresholding - Clean & Train Better Model

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=50,
    random_state=1
)


In [ ]:
sp500


In [ ]:
# STEP 11: Probability-Based Predictions
"""
STEP 11: Predict with Confidence

We only predict UP if probability ≥ 60%
"""

def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    
    preds = model.predict_proba(test[predictors])[:,1]
    preds[preds >= 0.6] = 1
    preds[preds < 0.6] = 0
    
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined


In [ ]:
# STEP 12: Final Backtest & Evaluation
predictions = backtest(sp500, model, new_predictors)

predictions["Predictions"].value_counts()

precision_score(
    predictions["Target"],
    predictions["Predictions"]
)

predictions["Target"].value_counts() / predictions.shape[0]


In [ ]:
predictions


In [ ]:
"""
STEP 8: Trading Strategy Evaluation - Strategy vs Buy-and-Hold
"""

returns = sp500.loc[predictions.index, "Close"].pct_change()
strategy_returns = returns * predictions["Predictions"]

pd.DataFrame({
    "Market": returns.cumsum(),
    "Strategy": strategy_returns.cumsum()
}).plot(title="Strategy vs Market")


In [ ]:
import joblib

# Save trained model
joblib.dump(model, "sp500_model.pkl")

# Save predictors list
joblib.dump(new_predictors, "predictors.pkl")
"""
FastAPI app for S&P 500 ML model inference
"""

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd

# Load model and predictors
model = joblib.load("sp500_model.pkl")
predictors = joblib.load("predictors.pkl")

# Create FastAPI app
app = FastAPI(title="S&P 500 ML Prediction API")

# -----------------------------
# Input schema (data contract)
# -----------------------------
class MarketFeatures(BaseModel):
    Open: float
    High: float
    Low: float
    Close: float
    Volume: float

    # Rolling features (examples)
    Close_Ratio_2: float
    Trend_2: float
    Close_Ratio_5: float
    Trend_5: float
    Close_Ratio_60: float
    Trend_60: float
    Close_Ratio_250: float
    Trend_250: float
    Close_Ratio_1000: float
    Trend_1000: float


# -----------------------------
# Prediction endpoint
# -----------------------------
@app.post("/predict")
def predict_market(data: MarketFeatures):
    """
    Returns market direction prediction:
    1 = Market UP
    0 = Market DOWN
    """

    df = pd.DataFrame([data.dict()])

    probability = model.predict_proba(df[predictors])[0][1]
    prediction = int(probability >= 0.6)

    return {
        "prediction": prediction,
        "confidence": round(probability, 3)
    }



In [ ]:
"""
==============================
PROJECT UNDERSTANDING & BRIEFING
==============================

PROBLEM STATEMENT
-----------------
The goal of this project is to predict whether the S&P 500 index
will go UP or DOWN on the next trading day using historical price data.

APPROACH
--------
1. Collected historical S&P 500 data using Yahoo Finance
2. Cleaned and prepared time-series data
3. Engineered meaningful features:
   - Rolling price ratios
   - Short-term and long-term market trends
4. Built a Random Forest classification model
5. Evaluated performance using walk-forward backtesting
6. Optimized for precision to reduce false trading signals

WHY THIS APPROACH IS REALISTIC
------------------------------
- Uses only past data (no data leakage)
- Mimics real-world trading decisions
- Backtesting simulates live deployment
- Feature engineering reflects market behavior

KEY LEARNINGS
-------------
- Time-series data must be handled differently from normal ML data
- Feature engineering is more important than model complexity
- Backtesting is critical for real-world validation
- Higher accuracy does NOT always mean better trading performance

LIMITATIONS
-----------
- Does not include transaction costs or slippage
- Does not incorporate news or macroeconomic indicators
- Predicts direction, not magnitude of returns

FUTURE IMPROVEMENTS
-------------------
- Add trading cost simulation
- Compare multiple ML models
- Incorporate volatility-adjusted position sizing
- Add macroeconomic indicators
- Deploy as a live prediction pipeline

FINAL NOTE
----------
This project demonstrates an end-to-end machine learning workflow:
from data collection and feature engineering to evaluation and analysis.
"""
